In [1]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

In [2]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],
        [0.55, 0.87, 0.66],
        [0.57, 0.85, 0.64],
        [0.22, 0.58, 0.33],
        [0.77, 0.25, 0.10],
        [0.05, 0.8, 0.55]
    ]
)
attention_scores = torch.empty(inputs.shape[0])
query = inputs[1]
for idx, input in enumerate(inputs):
    attention_scores[idx] = torch.dot(input, query)

print(attention_scores)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [3]:
attention_weights = torch.softmax(attention_scores, dim=0)
print(attention_weights)
print(attention_weights.sum())

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.)


In [4]:
Z2 = torch.zeros(query.shape)
for i, input in enumerate(inputs):
    Z2 += attention_weights[i] * input
print(Z2)

tensor([0.4419, 0.6515, 0.5683])


In [5]:
print(inputs.shape)

torch.Size([6, 3])


In [6]:
attention_scores = inputs @ inputs.T

In [7]:
print(attention_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [8]:
attention_weights = torch.softmax(attention_scores, dim=-1)
print(attention_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [9]:
all_contexts = attention_weights @ inputs
print(all_contexts)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [10]:
in_dim = inputs.shape[1]
out_dim = 2
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(in_dim, out_dim), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(in_dim, out_dim), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(in_dim, out_dim), requires_grad=False)

In [11]:
x_2 = inputs[1]
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)
print(key_2)
print(value_2)

tensor([0.4306, 1.4551])
tensor([0.4433, 1.1419])
tensor([0.3951, 1.0037])


In [12]:
keys = inputs @ W_key
print(keys)
values = inputs @ W_value
print(values)

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]])
tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]])


In [15]:
class SelfAttention(torch.nn.Module):
    def __init__(self, in_dim, out_dim, bias=True):
        super().__init__()
        self.W_query = nn.Linear(in_dim, out_dim, bias=bias)
        self.W_key = nn.Linear(in_dim, out_dim, bias=bias)
        self.W_value = nn.Linear(in_dim, out_dim, bias=bias)

    def forward(self, x):
        query = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        x_attention_scores = query @ keys.T
        x_attention_weights = torch.softmax(x_attention_scores / keys.shape[-1] ** 0.5, dim=-1)

        x_context = x_attention_weights @ values
        return x_context


In [16]:
selfAttention = SelfAttention(in_dim, out_dim)
print(selfAttention(inputs))

tensor([[-0.4392, -0.2854],
        [-0.4403, -0.2886],
        [-0.4403, -0.2885],
        [-0.4401, -0.2885],
        [-0.4396, -0.2874],
        [-0.4403, -0.2889]], grad_fn=<MmBackward0>)


In [17]:
dropout = torch.nn.Dropout(0.5)
random_inputs = torch.rand(6, 6)
print(dropout(random_inputs))


tensor([[0.7668, 0.8901, 0.0251, 0.0000, 0.0000, 1.6112],
        [0.2919, 0.0000, 1.4153, 0.0000, 0.0000, 0.0000],
        [0.0000, 1.7052, 1.4639, 1.0365, 1.1966, 0.0000],
        [0.4502, 0.6221, 0.3910, 0.0000, 0.0000, 1.3497],
        [0.2331, 1.7716, 0.0000, 1.6918, 0.6066, 0.0000],
        [1.9764, 1.6727, 0.0000, 0.0000, 0.0000, 0.0000]])
